# 🍈 Clustering Wilayah Rawan Bencana Indonesia

**Dataset:** BNPB Records (2018-2024)  
**Metode:** K-Means Clustering  
**Author:** Guava

---

## 📋 Tujuan Project

Mengelompokkan provinsi di Indonesia berdasarkan pola bencana alam untuk:
1. Identifikasi zona risiko bencana
2. Rekomendasi alokasi resources BNPB
3. Prioritas mitigasi per wilayah

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded")

## 2️⃣ Load Data

In [ ]:
# Load dataset
df = pd.read_excel('data_bencana.xlsx')

print(f"📊 Data shape: {df.shape}")
print(f"\n📋 Columns: {df.columns.tolist()}")

# Display sample
df.head()

In [ ]:
# Info dataset
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

## 3️⃣ Data Preprocessing

In [ ]:
# Check missing values
print("❓ Missing Values:")
print(df.isnull().sum())

In [ ]:
# Handle missing values
df.dropna(subset=['death'], inplace=True)
df['missing_person'].fillna(0, inplace=True)
df['injured_person'].fillna(0, inplace=True)
df['flooded_house'].fillna(0, inplace=True)
df['damaged_house'].fillna(0, inplace=True)
df['damaged_facility'].fillna(0, inplace=True)

print(f"✅ Data after cleaning: {len(df)} rows")
print(f"✅ Missing values handled")

In [ ]:
# Extract date features
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print("✅ Date features extracted")
print(f"\n📅 Periode data: {df['date'].min()} s/d {df['date'].max()}")

## 4️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# Top 10 jenis bencana
plt.figure(figsize=(12, 6))
disaster_counts = df['disaster_type'].value_counts()
disaster_counts.plot(kind='barh', color='coral', edgecolor='black')
plt.title('Distribusi Jenis Bencana di Indonesia (2018-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Jumlah Kejadian', fontsize=12)
plt.ylabel('Jenis Bencana', fontsize=12)
plt.tight_layout()
plt.show()

print(disaster_counts)

In [ ]:
# Top 10 provinsi paling rawan
plt.figure(figsize=(12, 6))
province_counts = df['province'].value_counts().head(10)
province_counts.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Top 10 Provinsi dengan Kejadian Bencana Terbanyak', fontsize=16, fontweight='bold')
plt.xlabel('Jumlah Kejadian', fontsize=12)
plt.ylabel('Provinsi', fontsize=12)
plt.tight_layout()
plt.show()

print(province_counts)

In [ ]:
# Trend bencana per tahun
plt.figure(figsize=(12, 6))
year_counts = df['year'].value_counts().sort_index()
year_counts.plot(kind='line', marker='o', color='green', linewidth=2, markersize=8)
plt.title('Trend Kejadian Bencana per Tahun (2018-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Jumlah Kejadian', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5️⃣ Feature Engineering

In [ ]:
# Agregasi per provinsi
province_agg = df.groupby('province').agg({
    'city_id': 'count',
    'death': 'sum',
    'injured_person': 'sum',
    'missing_person': 'sum',
    'damaged_house': 'sum',
    'flooded_house': 'sum',
    'damaged_facility': 'sum'
}).reset_index()

province_agg.columns = ['province', 'total_disasters', 'total_death', 'total_injured', 
                        'total_missing', 'total_damaged_house', 'total_flooded_house', 
                        'total_damaged_facility']

print(f"✅ Agregasi provinsi: {len(province_agg)} rows")
province_agg.head()

In [ ]:
# Pivot jenis bencana per provinsi
disaster_pivot = df.pivot_table(
    index='province', 
    columns='disaster_type', 
    values='city_id', 
    aggfunc='count', 
    fill_value=0
).reset_index()

print(f"✅ Disaster pivot: {disaster_pivot.shape}")
disaster_pivot.head()

In [ ]:
# Merge untuk feature matrix lengkap
province_features = province_agg.merge(disaster_pivot, on='province')

print(f"✅ Feature matrix: {province_features.shape}")
print(f"   {len(province_features)} provinsi × {len(province_features.columns)-1} features")

# Save
province_features.to_csv('clustering_province_features.csv', index=False)
print("💾 Saved: clustering_province_features.csv")

province_features.head()

## 6️⃣ Data Standardization

In [ ]:
# Prepare features untuk clustering
features_to_cluster = province_features.drop('province', axis=1)

# Standardize
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_to_cluster)

print(f"✅ Features standardized: {features_scaled.shape}")

## 7️⃣ Elbow Method & Silhouette Score

In [ ]:
# Calculate inertia & silhouette scores
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(features_scaled, kmeans.labels_))

print("✅ Elbow method calculated")

In [ ]:
# Plot Elbow & Silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia', fontsize=12)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Silhouette plot
ax2.plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Saved: elbow_method.png")

In [ ]:
# Print scores
print("📈 Silhouette Scores:")
for k, score in zip(K_range, silhouette_scores):
    print(f"  K={k}: {score:.4f}")

optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\n🎯 Optimal K: {optimal_k} (Score: {max(silhouette_scores):.4f})")

## 8️⃣ K-Means Clustering

In [ ]:
# Clustering dengan K optimal
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
province_features['cluster'] = kmeans.fit_predict(features_scaled)

print(f"✅ Clustering completed with K={optimal_k}")
print(f"\n📊 Cluster distribution:")
print(province_features['cluster'].value_counts().sort_index())

## 9️⃣ Relabel Cluster (0=Low, 1=Extreme, 2=High)

In [ ]:
# Calculate severity score per cluster
cluster_severity = province_features.groupby('cluster').agg({
    'total_disasters': 'mean',
    'total_death': 'mean',
    'total_injured': 'mean',
    'total_damaged_house': 'mean'
})

# Normalize and compute composite score
norm_scaler = MinMaxScaler()
normalized = norm_scaler.fit_transform(cluster_severity)
cluster_severity['composite_score'] = normalized.mean(axis=1)

print("📈 Severity scores:")
print(cluster_severity[['composite_score']].sort_values('composite_score'))

In [ ]:
# Map old to new cluster
severity_rank = cluster_severity['composite_score'].rank().astype(int) - 1
old_to_new = severity_rank.to_dict()

print("🔄 Cluster mapping:")
for old, new in old_to_new.items():
    print(f"  Cluster {old} (old) → Cluster {new} (new)")

# Apply relabeling
province_features['cluster'] = province_features['cluster'].map(old_to_new)

print("\n✅ Clusters relabeled (0=Low, 1=Extreme, 2=High)")

## 🔟 Cluster Analysis

In [ ]:
# Detailed analysis per cluster
for cluster_id in range(optimal_k):
    cluster_data = province_features[province_features['cluster'] == cluster_id]
    
    print("="*70)
    print(f"CLUSTER {cluster_id} - {len(cluster_data)} provinsi")
    print("="*70)
    
    print(f"\n📍 Provinsi:")
    for prov in cluster_data['province'].tolist():
        print(f"  - {prov}")
    
    print(f"\n📊 Rata-rata Metrics:")
    print(f"  Total Bencana: {cluster_data['total_disasters'].mean():.2f}")
    print(f"  Korban Meninggal: {cluster_data['total_death'].mean():.2f}")
    print(f"  Korban Luka: {cluster_data['total_injured'].mean():.2f}")
    print(f"  Rumah Rusak: {cluster_data['total_damaged_house'].mean():.2f}")
    print("\n")

In [ ]:
# Save results
province_features.to_csv('clustering_results.csv', index=False)
print("💾 Saved: clustering_results.csv")

## 1️⃣1️⃣ PCA Visualization

In [ ]:
# PCA for 2D visualization
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features_scaled)

province_features['PCA1'] = features_pca[:, 0]
province_features['PCA2'] = features_pca[:, 1]

print(f"📈 Explained Variance:")
print(f"  PCA1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"  PCA2: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"  Total: {sum(pca.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# Plot PCA scatter
plt.figure(figsize=(16, 10))

colors = ['#44AA44', '#FFA500', '#FF4444']
cluster_names = ['Cluster 0: LOW RISK', 'Cluster 1: EXTREME RISK', 'Cluster 2: HIGH RISK']
markers = ['o', '^', 's']

for i in range(optimal_k):
    cluster_data = province_features[province_features['cluster'] == i]
    plt.scatter(cluster_data['PCA1'], cluster_data['PCA2'], 
                c=colors[i], label=cluster_names[i], s=250, alpha=0.7, 
                edgecolors='black', linewidth=2, marker=markers[i])
    
    for idx, row in cluster_data.iterrows():
        plt.annotate(row['province'], 
                    (row['PCA1'], row['PCA2']),
                    fontsize=9, alpha=0.8, ha='center', fontweight='bold')

plt.xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', 
          fontsize=14, fontweight='bold')
plt.ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', 
          fontsize=14, fontweight='bold')
plt.title('Clustering Wilayah Rawan Bencana Indonesia\n(0=Low Risk, 1=Extreme Risk, 2=High Risk)', 
          fontsize=18, fontweight='bold', pad=20)
plt.legend(fontsize=13, loc='best', frameon=True, shadow=True)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('clustering_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Saved: clustering_visualization.png")

## 1️⃣2️⃣ Cluster Comparison

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Perbandingan Karakteristik Cluster\n(0=Low, 1=Extreme, 2=High)', 
             fontsize=18, fontweight='bold', y=0.995)

colors_cluster = ['#44AA44', '#FFA500', '#FF4444']

# 1. Total Bencana
cluster_avg = province_features.groupby('cluster')['total_disasters'].mean()
axes[0, 0].bar(cluster_avg.index, cluster_avg.values, color=colors_cluster, 
              edgecolor='black', linewidth=2)
axes[0, 0].set_xlabel('Cluster', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Rata-rata Total Bencana', fontsize=12)
axes[0, 0].set_title('Total Kejadian Bencana', fontsize=14, fontweight='bold')
axes[0, 0].set_xticks([0, 1, 2])
axes[0, 0].set_xticklabels(['0\n(Low)', '1\n(Extreme)', '2\n(High)'])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(cluster_avg.values):
    axes[0, 0].text(i, v + 100, f'{v:.0f}', ha='center', fontweight='bold', fontsize=12)

# 2. Korban Jiwa
cluster_death = province_features.groupby('cluster')['total_death'].mean()
axes[0, 1].bar(cluster_death.index, cluster_death.values, color=colors_cluster, 
              edgecolor='black', linewidth=2)
axes[0, 1].set_xlabel('Cluster', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Rata-rata Korban Meninggal', fontsize=12)
axes[0, 1].set_title('Korban Meninggal', fontsize=14, fontweight='bold')
axes[0, 1].set_xticks([0, 1, 2])
axes[0, 1].set_xticklabels(['0\n(Low)', '1\n(Extreme)', '2\n(High)'])
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(cluster_death.values):
    axes[0, 1].text(i, v + 150, f'{v:.0f}', ha='center', fontweight='bold', fontsize=12)

# 3. Rumah Rusak
cluster_damaged = province_features.groupby('cluster')['total_damaged_house'].mean()
axes[1, 0].bar(cluster_damaged.index, cluster_damaged.values, color=colors_cluster, 
              edgecolor='black', linewidth=2)
axes[1, 0].set_xlabel('Cluster', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Rata-rata Rumah Rusak', fontsize=12)
axes[1, 0].set_title('Kerusakan Rumah', fontsize=14, fontweight='bold')
axes[1, 0].set_xticks([0, 1, 2])
axes[1, 0].set_xticklabels(['0\n(Low)', '1\n(Extreme)', '2\n(High)'])
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(cluster_damaged.values):
    axes[1, 0].text(i, v + 3000, f'{v:.0f}', ha='center', fontweight='bold', fontsize=12)

# 4. Jumlah Provinsi
cluster_count = province_features['cluster'].value_counts().sort_index()
axes[1, 1].bar(cluster_count.index, cluster_count.values, color=colors_cluster, 
              edgecolor='black', linewidth=2)
axes[1, 1].set_xlabel('Cluster', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Jumlah Provinsi', fontsize=12)
axes[1, 1].set_title('Distribusi Provinsi', fontsize=14, fontweight='bold')
axes[1, 1].set_xticks([0, 1, 2])
axes[1, 1].set_xticklabels(['0\n(Low)', '1\n(Extreme)', '2\n(High)'])
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(cluster_count.values):
    axes[1, 1].text(i, v + 0.8, f'{v}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('cluster_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Saved: cluster_comparison.png")

## 1️⃣3️⃣ Heatmap Top 15 Provinces

In [ ]:
# Top 15 provinces heatmap
top15 = province_features.nlargest(15, 'total_disasters')[
    ['province', 'cluster', 'total_disasters', 'total_death', 
     'total_injured', 'total_damaged_house']
].copy()

# Normalize
top15_normalized = top15.copy()
for col in ['total_disasters', 'total_death', 'total_injured', 'total_damaged_house']:
    top15_normalized[col] = (top15[col] - top15[col].min()) / (top15[col].max() - top15[col].min())

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(top15_normalized.set_index('province')[
    ['total_disasters', 'total_death', 'total_injured', 'total_damaged_house']
], annot=False, cmap='YlOrRd', cbar_kws={'label': 'Normalized Value'}, 
linewidths=0.5, linecolor='white')

plt.title('Heatmap: Top 15 Provinsi Paling Rawan Bencana (Normalized)', 
          fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Metrics', fontsize=12, fontweight='bold')
plt.ylabel('Provinsi', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('province_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Saved: province_heatmap.png")

## ✅ Summary

### Hasil Clustering:

- **Cluster 0 (Low Risk):** 36 provinsi dengan risiko moderat
- **Cluster 1 (Extreme Risk):** Sulawesi Tengah dengan korban jiwa tertinggi
- **Cluster 2 (High Risk):** Jawa Barat, Jawa Tengah, Jawa Timur dengan frekuensi bencana tertinggi

### Output Files:

1. `clustering_province_features.csv` - Feature matrix
2. `clustering_results.csv` - Hasil clustering
3. `elbow_method.png` - Grafik penentuan K optimal
4. `clustering_visualization.png` - PCA scatter plot
5. `cluster_comparison.png` - Bar chart comparison
6. `province_heatmap.png` - Heatmap top 15 provinsi

---

🍈 **Project by Guava - Clustering Wilayah Rawan Bencana Indonesia**